# M2 N/V 축 차원·개입 강도 1시드 탐색

기존 seed 42의 M1과 `axis_dim=4, rho=0.05` M2 결과는 다시 학습하지 않고 Drive에서 읽습니다. 아래 두 M2만 한 런타임에서 순서대로 seed 42, 300 epoch까지 학습합니다.

1. `axis_wide`: `axis_dim=8`, `rho=0.05`
2. `weak_signal`: `axis_dim=4`, `rho=0.025`

ID 64차원, L2 `1e-3`, 2층, K=1 균일 음성표본, 이진 그래프, 일반 BPR은 고정합니다. Dunnhumby 개발 구간 DAY 684~690만 사용하며, 최종 test·holdout은 생성하지 않습니다. 단일 시드 결과는 후보 탐색용이며 유의성·일반화·최종 선택을 주장하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REVIEWED_SHA = '602fef1d1546d86f92adc5cb41f89059d7d599be'
%cd /content
!rm -rf /content/clv-m2-lightgcn-runner
!git clone -q https://github.com/jung-un/clv-m2-lightgcn-runner.git
%cd /content/clv-m2-lightgcn-runner
!git checkout -q $REVIEWED_SHA
import subprocess
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip() == REVIEWED_SHA


In [ ]:
from pathlib import Path
import json
import torch
import clv_m2_axis_rho_seed42_screen as screen

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
ROOT = Path('/content/drive/MyDrive/논문/data')
OUT_DIR = ROOT / 'results_v3_dunnhumby_clv_m2_capacity_search_v1'
cfg = screen.configure(out_dir=str(OUT_DIR))
summary = screen.preflight_summary(cfg)
assert summary['execution_order'] == [
    {'condition': 'axis_wide', 'axis_dim': 8, 'rho': 0.05},
    {'condition': 'weak_signal', 'axis_dim': 4, 'rho': 0.025},
]
assert summary['seed'] == 42 and summary['fixed_epoch'] == 300
assert summary['final_test_constructed'] is False
assert summary['holdout_constructed'] is False
print(json.dumps(summary, ensure_ascii=False, indent=2))


In [ ]:
# 기준 결과가 누락·불일치하면 신규 학습 전에 중단합니다.
# 정상이면 axis_dim=8 → rho=0.025 순으로 자동 실행합니다.
paths = screen.run(cfg, root=ROOT)
print(json.dumps(paths, ensure_ascii=False, indent=2))


In [ ]:
import pandas as pd
from IPython.display import display

absolute = pd.read_csv(paths['absolute'])
comparison = pd.read_csv(paths['comparison'])
reading = pd.read_csv(paths['reading'])
headline_metrics = [
    'recall@10', 'ndcg@10', 'price_purchase_amount_weighted_hit@10',
    'vndcg@10', 'recall@50', 'price_purchase_amount_weighted_hit@50',
]
print('1) 300 epoch 전체 absolute')
display(absolute)
print('2) M1 및 기존 axis4/rho.05 대비 핵심 차이')
display(comparison[comparison.metric.isin(headline_metrics)])
print('3) 전체 지표 비교는 CSV에 누락 없이 저장됩니다.')
print('4) 단일 시드 탐색 판독')
display(reading)


In [ ]:
# 런타임에서 로컬로 내려받을 결과 묶음
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

zip_path = '/content/clv_m2_axis_rho_seed42_screen_results.zip'
with ZipFile(zip_path, 'w', ZIP_DEFLATED) as archive:
    for name, path in paths.items():
        archive.write(path, arcname=Path(path).name)
files.download(zip_path)
